In [ ]:
# # AI-Based Cardiovascular Disease Prediction using Hybrid Quantum Neural Networks

# ## Notebook 2 : CNN Feature Extractor

# ### Objectives

# - Load processed ECG dataset
# - Encode heartbeat labels
# - Perform patient-wise train/validation/test split
# - Build Residual 1D CNN
# - Train CNN
# - Save best CNN model
# - Extract 16-dimensional learned features

In [3]:
# ==============================================================================
# HOSPITAL SANDBOX RUN: FIXED END-TO-END DATA ENVELOPE & 1D-CNN TRAINING
# ==============================================================================

# 1. Mount Google Drive First
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 2. Install and Import Libraries
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

# Set random seeds for strict reproducible training loops
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# ==============================================================================
# STEP 3: ESTABLISH SANDBOX VERIFIED DIRECTORY PATHS
# ==============================================================================
SANDBOX_DIR = "/content/drive/MyDrive/(1)Hybrid_QML_CVD_Project_HospitalRun"
DATASETS_DIR = os.path.join(SANDBOX_DIR, "datasets")
FEATURES_DIR = os.path.join(SANDBOX_DIR, "features")
MODELS_DIR = os.path.join(SANDBOX_DIR, "models")

# Ensure all structural output directories exist inside your sandbox
os.makedirs(FEATURES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# Define explicit categorical index definitions matching your project structure
class_mapping = {"Normal": 0, "LBBB": 1, "RBBB": 2, "APB": 3, "PVC": 4}

# ==============================================================================
# STEP 4: DATA LOAD, DYNAMIC TRIMMING, AND LABEL COERCION ENGINE
# ==============================================================================
print("🔄 Loading multi-domain data arrays inside the sandbox workspace...")

try:
    # Load baseline MIT-BIH dataset arrays from your sandbox copy
    mit_beats = np.load(os.path.join(DATASETS_DIR, "beats.npy"))
    mit_labels = np.load(os.path.join(DATASETS_DIR, "labels.npy"))

    # Load your new 12,057 segmented hospital arrays from your sandbox copy
    hospital_beats = np.load(os.path.join(DATASETS_DIR, "hospital_beats.npy"))
    hospital_labels = np.load(os.path.join(DATASETS_DIR, "hospital_labels.npy"))

    print(f"📏 Initial Shapes -> MIT-BIH: {mit_beats.shape} | Hospital: {hospital_beats.shape}")

    # 1. Slice configurations to 180 length elements
    if mit_beats.shape[1] == 200 and hospital_beats.shape[1] == 180:
        print("✂️ Trimming MIT-BIH beats from 200 down to 180 to align center configurations...")
        mit_beats = mit_beats[:, 10:190]
    elif mit_beats.shape[1] == 180 and hospital_beats.shape[1] == 200:
        print("✂️ Trimming Hospital beats from 200 down to 180 to align center configurations...")
        hospital_beats = hospital_beats[:, 10:190]

    # Flatten inputs to avoid nested array dimensional warnings
    if mit_labels.ndim > 1: mit_labels = mit_labels.flatten()
    if hospital_labels.ndim > 1: hospital_labels = hospital_labels.flatten()

    # 2. STRING LABEL CONVERSION FIX: Coerce any leftover text characters back to numeric codes
    cleaned_mit_labels = []
    for lbl in mit_labels:
        # If the loaded label is an encoded byte string, decode it
        if isinstance(lbl, bytes):
            lbl = lbl.decode('utf-8')
        lbl_str = str(lbl).strip()

        # Check if it's already a numeric digit character like '0'
        if lbl_str.isdigit():
            cleaned_mit_labels.append(int(lbl_str))
        else:
            # Map structural text arrays directly to target integers
            cleaned_mit_labels.append(class_mapping.get(lbl_str, 0))

    mit_labels_numeric = np.array(cleaned_mit_labels, dtype=np.int64)
    hospital_labels_numeric = hospital_labels.astype(np.int64)

    # 3. Combine both pure numeric vectors safely into an aggregate pool
    X_combined = np.concatenate((mit_beats, hospital_beats), axis=0)
    y_combined = np.concatenate((mit_labels_numeric, hospital_labels_numeric), axis=0)

    # Add explicit channel dimension for Conv1D compatibility (Batch, Channels, Length)
    X_combined = np.expand_dims(X_combined, axis=1)

    print("\n" + "="*65)
    print(" 📊 SANDBOX UNIFIED MATRIX COMPILATION DETAILS")
    print("="*65)
    print(f" 🔹 Aligned MIT-BIH Beats Loaded : {len(mit_beats)}")
    print(f" 🔹 Local Hospital Slices Injected: {len(hospital_beats)}")
    print(f" 🔹 Aggregated Array Training Pool: {len(X_combined)}")
    print(f" 🔹 Tensor Input Feature Structure: {X_combined.shape} (Batch, Channel, Width)")
    print("="*65)

except FileNotFoundError as e:
    raise FileNotFoundError(f"\n❌ Path alignment error: Verify files are present in Drive.\n{str(e)}")

# ==============================================================================
# STEP 5: TRAIN-TEST SPLIT AND PYTORCH DATA SET-UP
# ==============================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y_combined, test_size=0.2, random_state=42, stratify=y_combined
)

# Convert arrays into explicit PyTorch floating/long tensor variables
train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long))

# Instantiate performance-optimized batch loaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# ==============================================================================
# STEP 6: RECONSTRUCT 1D-CNN NETWORK MODEL ARCHITECTURE
# ==============================================================================
class ECG1DCNNFeatureExtractor(nn.Module):
    def __init__(self, input_length=180, feature_dim=16):
        super(ECG1DCNNFeatureExtractor, self).__init__()
        self.conv_block = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2),  # Length reduces to 90

            nn.Conv1d(16, 32, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),  # Length reduces to 45
            nn.Flatten()
        )
        self.fc_out = nn.Linear(32 * 45, feature_dim)
        self.classifier = nn.Linear(feature_dim, 5)

    def forward(self, x):
        features = self.conv_block(x)
        features = self.fc_out(features)
        logits = self.classifier(features)
        return features, logits

# Bind execution engine to hardware accelerators if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ECG1DCNNFeatureExtractor(input_length=180, feature_dim=16).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# ==============================================================================
# STEP 7: MODEL TRAINING LOOP RUNNER
# ==============================================================================
epochs = 10
print(f"\n🚀 Initiating Training Loop on Target Device: {device}...")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        _, logits = model(inputs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(logits, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = (correct / total) * 100
    print(f" 🟢 Epoch [{epoch+1:02d}/{epochs:02d}] ➔ Loss: {epoch_loss:.4f} | Training Accuracy: {epoch_acc:.2f}%")

# Save model checkpoint weights securely to your sandbox folder copy
torch.save(model.state_dict(), os.path.join(MODELS_DIR, "best_model.pt"))
print("💾 Model Weight Checkpoint Artifacts Saved Successfully!")

# ==============================================================================
# STEP 8: EXTRACT AND EXPORT COMBINED LATENT LAYER FEATURES FOR THE QNN STEP
# ==============================================================================
print("\n🎬 Extracting intermediate latent vectors for Quantum Variational Layer processing...")
model.eval()

all_features = []
all_labels = []

full_eval_dataset = TensorDataset(torch.tensor(X_combined, dtype=torch.float32), torch.tensor(y_combined, dtype=torch.long))
full_loader = DataLoader(full_eval_dataset, batch_size=64, shuffle=False)

with torch.no_grad():
    for inputs, labels in full_loader:
        inputs = inputs.to(device)
        latent_features, _ = model(inputs)
        all_features.append(latent_features.cpu().numpy())
        all_labels.append(labels.numpy())

cnn_features = np.concatenate(all_features, axis=0)
cnn_labels = np.concatenate(all_labels, axis=0)

np.save(os.path.join(FEATURES_DIR, "cnn_features.npy"), cnn_features)
np.save(os.path.join(FEATURES_DIR, "cnn_labels.npy"), cnn_labels)

print("\n" + "="*65)
print(" 🎉 SUCCESS: UNIFIED FEATURE PIPELINE COMPLETED")
print("="*65)
print(f" 🔹 Saved Features Binary File Shape : {cnn_features.shape}")
print(f" 🔹 Saved Labels Binary File Shape   : {cnn_labels.shape}")
print(f" 🔹 Target Output Feature Folder     : {FEATURES_DIR}")
print("="*65)

Mounted at /content/drive
🔄 Loading multi-domain data arrays inside the sandbox workspace...
📏 Initial Shapes -> MIT-BIH: (100106, 200) | Hospital: (12057, 180)
✂️ Trimming MIT-BIH beats from 200 down to 180 to align center configurations...

 📊 SANDBOX UNIFIED MATRIX COMPILATION DETAILS
 🔹 Aligned MIT-BIH Beats Loaded : 100106
 🔹 Local Hospital Slices Injected: 12057
 🔹 Aggregated Array Training Pool: 112163
 🔹 Tensor Input Feature Structure: (112163, 1, 180) (Batch, Channel, Width)

🚀 Initiating Training Loop on Target Device: cpu...
 🟢 Epoch [01/10] ➔ Loss: 0.2563 | Training Accuracy: 91.14%
 🟢 Epoch [02/10] ➔ Loss: 0.1990 | Training Accuracy: 92.77%
 🟢 Epoch [03/10] ➔ Loss: 0.1865 | Training Accuracy: 93.14%
 🟢 Epoch [04/10] ➔ Loss: 0.1772 | Training Accuracy: 93.41%
 🟢 Epoch [05/10] ➔ Loss: 0.1712 | Training Accuracy: 93.62%
 🟢 Epoch [06/10] ➔ Loss: 0.1662 | Training Accuracy: 93.73%
 🟢 Epoch [07/10] ➔ Loss: 0.1613 | Training Accuracy: 93.95%
 🟢 Epoch [08/10] ➔ Loss: 0.1580 | Trai